In [26]:
# -------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------

PRODUCTION_DB_PATH = "../web/data/Track.db"
ALLOW_DB_WRITES    = True                           # Safety guard: must be True to write to Track.db
DB_PATH            = PRODUCTION_DB_PATH             # Scraper writes directly to Track.db
YEAR               = 2026
GENDER             = "Girls"                        # "Boys" or "Girls"
MEET_TYPE          = "Sectional"                    # "Sectional", "Regional", or "State"
SEC_TO_PROCESS     = list(range(1, 33))             # Sectionals 1-32
REG_TO_PROCESS     = list(range(1, 9))              # Regionals 1-8
INDEX_MONTHS       = [5, 6]                         # Search both months for robustness
IHSAA_FROM_YEAR    = 2026                           # Use IHSAA round pages for this year and newer
MIN_ROWS_EXPECTED  = 30                             # Warning threshold for suspiciously low parses

# -------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------

import importlib
import requests
import re
import gc
import logging
import time
import random
import shutil
import sqlite3
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin
from pathlib import Path
import util.db_util as db_util_module
import util.conversion_util as conversion_util_module

db_util_module = importlib.reload(db_util_module)
conversion_util_module = importlib.reload(conversion_util_module)
Database = db_util_module.Database
Conversion = conversion_util_module.Conversion

# -------------------------------------------------------------
# CONSTANTS
# -------------------------------------------------------------

BASE    = "https://in.milesplit.com"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

FIELD_EVENTS = {"High Jump", "Long Jump", "Triple Jump", "Shot Put", "Discus", "Pole Vault"}
RELAY_EVENTS = {"4 x 100 Relay", "4 x 400 Relay", "4 x 800 Relay"}
NO_MARK      = {"NT", "ND", "NH", "NM", "DNS", "DNF", "DQ", "SCR", "--"}
GRADES       = {"FR", "SO", "JR", "SR", "9", "10", "11", "12",
                "Freshman", "Sophomore", "Junior", "Senior"}

EVENT_ALIASES = {
    "100 meters":          "100 Meters",    "100 meter dash":      "100 Meters",
    "100 meter run":       "100 Meters",
    "200 meters":          "200 Meters",    "200 meter dash":      "200 Meters",
    "200 meter run":       "200 Meters",
    "400 meters":          "400 Meters",    "400 meter dash":      "400 Meters",
    "400 meter run":       "400 Meters",
    "800 meters":          "800 Meters",    "800 meter run":       "800 Meters",
    "1600 meters":         "1600 Meters",   "1600 meter run":      "1600 Meters",
    "mile run":            "1600 Meters",
    "3200 meters":         "3200 Meters",   "3200 meter run":      "3200 Meters",
    "2 mile run":          "3200 Meters",
    "110 meter hurdles":   "110 Hurdles",   "110 hurdles":         "110 Hurdles",
    "110m hurdles":        "110 Hurdles",   "110 meter hurdle":    "110 Hurdles",
    "100 meter hurdles":   "100 Hurdles",   "100 hurdles":         "100 Hurdles",
    "100m hurdles":        "100 Hurdles",
    "300 meter hurdles":   "300 Hurdles",   "300 hurdles":         "300 Hurdles",
    "300m hurdles":        "300 Hurdles",   "300 meter hurdle":    "300 Hurdles",
    "4 x 100 meter relay": "4 x 100 Relay", "4 x 100 relay":       "4 x 100 Relay",
    "4x100 meter relay":   "4 x 100 Relay", "4x100 relay":         "4 x 100 Relay",
    "4x100m relay":        "4 x 100 Relay", "4x100":               "4 x 100 Relay",
    "4 x 100":             "4 x 100 Relay", "4 x 100m relay":      "4 x 100 Relay",
    "4 x 400 meter relay": "4 x 400 Relay", "4 x 400 relay":       "4 x 400 Relay",
    "4x400 meter relay":   "4 x 400 Relay", "4x400 relay":         "4 x 400 Relay",
    "4x400m relay":        "4 x 400 Relay", "4x400":               "4 x 400 Relay",
    "4 x 400":             "4 x 400 Relay", "4 x 400m relay":      "4 x 400 Relay",
    "4 x 800 meter relay": "4 x 800 Relay", "4 x 800 relay":       "4 x 800 Relay",
    "4x800 meter relay":   "4 x 800 Relay", "4x800 relay":         "4 x 800 Relay",
    "4x800m relay":        "4 x 800 Relay", "4x800":               "4 x 800 Relay",
    "4 x 800":             "4 x 800 Relay", "4 x 800m relay":      "4 x 800 Relay",
    "high jump":           "High Jump",     "long jump":           "Long Jump",
    "triple jump":         "Triple Jump",   "shot put":            "Shot Put",
    "discus":              "Discus",        "discus throw":        "Discus",
    "pole vault":          "Pole Vault",
    "team scores":         "Team Scores",
}

_METRIC_RE        = re.compile(r"^\d+\.\d+m$", re.IGNORECASE)
_MARK_MIN_GAP     = 2
_TRAIL_TOKEN_RE   = re.compile(r"^[+\-]?\d{1,2}$|^[+\-]?\d+\.\d+$")
_RELAY_ATHLETE_RE = re.compile(r"\d+\)\s+((?:(?!\s*\d+\)).)+?)(?=\s{2,}\d+\)|\s*$)")

# -------------------------------------------------------------
# SETUP
# -------------------------------------------------------------

logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(message)s")
session = requests.Session()
session.headers.update(HEADERS)

# Resolve DB path and enforce an explicit write-safety switch before opening the database
if not ALLOW_DB_WRITES:
    raise RuntimeError("Set ALLOW_DB_WRITES = True to run scraper against Track.db")

DB_PATH = PRODUCTION_DB_PATH
prod_path = Path(DB_PATH).resolve()
backup_candidates = sorted(prod_path.parent.glob("Track_backup_*.db"), key=lambda p: p.name)

if not backup_candidates:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = prod_path.parent / f"Track_backup_{ts}.db"
    # Snapshot Track.db without moving it so the live DB path remains stable.
    shutil.copy2(prod_path, backup_path)
    print(f"Created initial backup: {backup_path.name}")
else:
    latest_backup = backup_candidates[-1]
    restore_ans = input(
        f"Backup exists ({latest_backup.name}). Type 'yes' to restore, or press Enter to continue without restoring: "
    ).strip().lower()
    if restore_ans == "yes":
        shutil.copy2(latest_backup, prod_path)
        print(f"Track.db restored from backup: {latest_backup.name}")
    else:
        print("Keeping current Track.db (no restore).")

print(f"Active DB: {DB_PATH}")

db = Database(DB_PATH)
convert = Conversion()
warningDF = pd.DataFrame(columns=["warning", "id", "desc"])
runStats = []

# -------------------------------------------------------------
# HELPERS
# -------------------------------------------------------------

def log_warning(warning, id_value, desc=""):
    if str(id_value) not in warningDF["id"].values:
        warningDF.loc[len(warningDF)] = {
            "warning": warning,
            "id": str(id_value),
            "desc": str(desc)
        }


def safe_get(url, retries=4, params=None):
    for attempt in range(retries + 1):
        time.sleep(1.5 + random.uniform(0, 1.5))
        try:
            r = session.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except Exception as e:
            if attempt == retries:
                log_warning("Fetch failed", url, str(e))
                return None
            time.sleep(min(10, 2 ** attempt))


def normalize_text(s):
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()


def grade_normalization(grade):
    return {
        "Freshman": "FR", "Fr": "FR", "FR": "FR", "9": "FR",
        "Sophomore": "SO", "So": "SO", "SO": "SO", "10": "SO",
        "Junior": "JR", "Jr": "JR", "JR": "JR", "11": "JR",
        "Senior": "SR", "Sr": "SR", "SR": "SR", "12": "SR"
    }.get(str(grade).strip(), "Unknown")


def calculate_grad_year(grade_raw):
    offset = {"SR": 0, "JR": 1, "SO": 2, "FR": 3}.get(grade_normalization(grade_raw), -1)
    return YEAR + offset if offset != -1 else 9999


def team_mapping(team):
    mappings = {
        # Raw-format truncations
        "Indianapolis Scecina Memo": "Indianapolis Scecina Memorial",
        "Greencastle - A":           "Greencastle",
        "Indianapolis Bishop Chata": "Indianapolis Bishop Chatard",
        "North Central (Indianapol": "North Central (Indianapolis)",
        "Indianapolis Arsenal Tech": "Indianapolis Arsenal Technical",
        "Indiana School for the De": "Indiana School for the Deaf",
        # Formatted API full names -> DB school_name
        "Cardinal Ritter High School":                   "Indianapolis Cardinal Ritter",
        "Crispus Attucks Medical Magnet High School":    "Indianapolis Crispus Attucks",
        "Covenant Christian High School (Indianapolis)": "Covenant Christian (Indpls)",
        "George Washington High School":                 "Indianapolis George Washington Community",
        "Christel House Watanabe Manual HS":             "Christel House",
        "Christel House Watanabe Manual":                "Christel House",
        # Sectional 21 API names
        "Cathedral High School":                        "Indianapolis Cathedral",
        "North Central High School (Indianapolis)":     "North Central (Indianapolis)",
        "Bishop Chatard Indianapolis":                  "Indianapolis Bishop Chatard",
        "Brebeuf Jesuit Prep School":                   "Brebeuf Jesuit Preparatory",
        "Arsenal Technical High School":                "Indianapolis Arsenal Technical",
        "Shortridge High School Indianapolis":          "Indianapolis Shortridge",
        "Purdue Polytechnic (Englewood)":               "Purdue Polytechnic - Downtown",
        "Purdue Polytechnic High School Broad Ripple":  "Purdue Polytechnic - Broad Ripple",
        "International High School":                    "International School of Indiana",
    }
    return mappings.get(team, team)


def normalize_event(raw):
    raw = re.sub(r"^(boys'?|girls'?)(\s+(boys|girls))?\s*[-\s]*", "", raw.strip(), flags=re.IGNORECASE)
    raw = re.sub(r"^(varsity|jv|junior varsity|freshman|frosh|sophomore|results)\s+", "", raw.strip(), flags=re.IGNORECASE)
    raw = re.sub(r"\s+(Flight|Heat|Section)\s+\d+$", "", raw.strip(), flags=re.IGNORECASE)
    raw = re.sub(r"\s+(Finals?|Preliminaries?|Prelims?)$", "", raw.strip(), flags=re.IGNORECASE)
    return EVENT_ALIASES.get(raw.lower().strip())


def _strip_trailing_heat_wind(tokens):
    _HEAT_RE = re.compile(r"^\d{1,2}$")
    _WIND_RE = re.compile(r"^[+\-]?\d+\.\d{1}$|^NWI$", re.IGNORECASE)
    while len(tokens) > 3:
        val = tokens[-1][2]
        if _HEAT_RE.match(val) or _WIND_RE.match(val):
            tokens = tokens[:-1]
        else:
            break
    return tokens


def _is_target_meet(anchor_text, meet_type, meet_num, gender):
    t = normalize_text(anchor_text)
    if "ihsaa" not in t:
        return False
    if gender.lower() not in t:
        return False

    mt = meet_type.lower()
    if mt == "state":
        return ("state" in t) and ("sectional" not in t) and ("regional" not in t)

    if mt not in t:
        return False

    num_token = str(meet_num)
    return bool(re.search(rf"\b{re.escape(num_token)}\b", t))


def _season_slug(start_year):
    return f"{start_year}-{(start_year + 1) % 100:02d}"


def _ihsaa_round_slug(meet_type):
    mt = (meet_type or "").strip().lower()
    if mt == "sectional":
        return "sectionals"
    if mt == "regional":
        return "regionals"
    return "state-finals"


def _ihsaa_tournament_url(year, gender, meet_type):
    gender_slug = "boys" if str(gender).strip().lower() == "boys" else "girls"
    season_slug = _season_slug(int(year))
    round_slug = _ihsaa_round_slug(meet_type)
    return f"https://www.ihsaa.org/sports/{gender_slug}/track-field/{season_slug}-tournament?round={round_slug}"


def _extract_ihsaa_num_host(text, default_num=None):
    s = re.sub(r"\s+", " ", text or "").strip()
    m = re.search(r"(?:^|\b)(\d{1,2})\.\s*([^|]+?)\s*\(", s)
    if m:
        return int(m.group(1)), m.group(2).strip()

    # Fallback for pages/rows that don't include numeric prefixes (most likely state finals).
    host = None
    m_host = re.search(r"\b([A-Z][A-Za-z'().\-]+(?:\s+[A-Z][A-Za-z'().\-]+){0,5})\s*\(", s)
    if m_host:
        host = m_host.group(1).strip()

    if default_num is not None:
        return int(default_num), host
    return None, host


def get_ihsaa_meet_links(meet_type, gender, year):
    url = _ihsaa_tournament_url(year, gender, meet_type)
    r = safe_get(url)
    if r is None:
        return {}

    soup = BeautifulSoup(r.text, "html.parser")
    links = {}
    default_num = 1 if (meet_type or "").strip().lower() == "state" else None

    for a in soup.find_all("a", href=True):
        href = a.get("href", "").strip()
        if "in.milesplit.com" not in href or "/results" not in href:
            continue

        meet_url = href if href.startswith("http") else urljoin(BASE, href)

        row_text = ""
        parent = a.parent
        if parent is not None:
            row_text = parent.get_text(" ", strip=True)
        if not row_text:
            row_text = a.get_text(" ", strip=True)

        meet_num, host = _extract_ihsaa_num_host(row_text, default_num=default_num)
        if meet_num is None:
            continue

        links[meet_num] = {
            "anchor_text": row_text,
            "meet_url": meet_url,
            "host": host,
            "source": "IHSAA"
        }

    return links


def _build_meet_info_from_meet_url(meet_url, host=None, anchor_text="", source="MileSplit"):
    r2 = safe_get(meet_url)
    if r2 is None:
        return {
            "raw_url": None,
            "raw_candidates": [],
            "host": host,
            "meet_url": meet_url,
            "anchor_text": anchor_text,
            "raw_option_found": False,
            "source": source
        }

    soup2 = BeautifulSoup(r2.text, "html.parser")
    seed_candidates = []
    raw_option_found = False

    select = soup2.find("select", id="ddResultsView")
    if select:
        for option in select.find_all("option"):
            val = option.get("value", "")
            if val and "?type=raw" in val:
                seed_candidates.append(val if val.startswith("http") else urljoin(BASE, val))
                raw_option_found = True

    seed_candidates.extend(_extract_raw_result_candidates(soup2))

    if not seed_candidates:
        seed_candidates.append(meet_url + "?type=raw")

    raw_candidates = _expand_raw_candidates(seed_candidates)

    return {
        "raw_url": (raw_candidates[0] if raw_candidates else None),
        "raw_candidates": raw_candidates,
        "host": host,
        "meet_url": meet_url,
        "anchor_text": anchor_text,
        "raw_option_found": raw_option_found,
        "source": source
    }



_ihsaa_meet_cache = {}

# -------------------------------------------------------------
# FORMAT A: STANDARD FIXED-WIDTH PARSER
# -------------------------------------------------------------

_EVENT_HEADER_RE = re.compile(
    r"^(?:Boys'?|Girls'?)[-\s]+(.+?)(?:\s+(Finals?|Preliminaries?|Prelims?))?$",
    re.IGNORECASE
)
_SKIP_SUBSECTION_RE     = re.compile(r"\b(Heat|Section|Flight)\s+\d+$", re.IGNORECASE)
_SEPARATOR_RE           = re.compile(r"^=+$")
_PLACE_RE               = re.compile(r"^\s*(\d+[=]?)\s+(.*)")
_RELAY_ATHLETE_SENTINEL = "__RELAY_ATHLETE__"


def detect_columns(header_line):
    if "Team" not in header_line:
        return None
    if "Time" not in header_line and "Mark" not in header_line:
        return None
    yr_start = header_line.find("Yr")
    team_start = header_line.find("Team")
    return {"yr": yr_start, "team": team_start}


def parse_row_fixed(line, cols, is_relay, is_field):
    row = line.rstrip()
    all_tokens = [(m.start(), m.end(), m.group()) for m in re.finditer(r"\S+", row)]
    if not all_tokens:
        return None, None, None, None

    tokens = [t for t in all_tokens if not _METRIC_RE.match(t[2])]
    if not tokens:
        return None, None, None, None

    tokens = _strip_trailing_heat_wind(tokens)

    _mark_p = (
        r"(\d+:\d{2}\.\d+)$"
        r"|(\d{1,2}\.\d{2,})$"
        r"|(\d{1,3}-\d{1,2}(?:\.\d+)?[\"'.]?)$"
        r"|(\d{1,3}['])$"
    )
    _MARK_SUFFIX_RE = re.compile(_mark_p)
    del _mark_p

    team_col = cols["team"] if cols and not is_relay else 5
    mark_earliest = team_col + 10

    mark_token = None
    for i in range(len(tokens) - 1, 1, -1):
        gap = tokens[i][0] - tokens[i - 1][1]
        val = tokens[i][2]
        if gap >= _MARK_MIN_GAP:
            if is_relay or val.upper() not in {g.upper() for g in GRADES}:
                mark_token = tokens[i]
                break

    if mark_token is None or (not is_relay and mark_token[2].upper() in {g.upper() for g in GRADES}):
        for t in tokens:
            if t[0] < mark_earliest:
                continue
            val = t[2]
            if val.upper() in {g.upper() for g in GRADES}:
                continue
            m = _MARK_SUFFIX_RE.search(val)
            if m:
                suffix = next(g for g in m.groups() if g is not None)
                mark_token = (t[0] + m.start(), t[1], suffix)
                break

    if mark_token is None:
        return None, None, None, None

    mark_start = mark_token[0]
    mark_val = mark_token[2]
    if mark_val.upper() in NO_MARK:
        return None, None, None, None
    if is_field:
        mark_val = mark_val.rstrip("\".")

    padded = row + " " * 10
    if is_relay:
        return "", "", padded[5:mark_start].strip(), mark_val

    name = padded[5:cols["yr"]].strip()
    grade = padded[cols["yr"]:cols["team"]].strip()
    team = padded[cols["team"]:mark_start].strip()
    if grade.upper() not in {g.upper() for g in GRADES}:
        grade = ""
    return name, grade, team, mark_val


def parse_format_a(text, label=""):
    records = []
    current_event = None
    current_type = None
    skip_block = False
    current_cols = None
    block_lines = []

    def flush():
        if not current_event or skip_block or current_cols is None:
            return
        is_relay = current_event in RELAY_EVENTS
        is_field = current_event in FIELD_EVENTS
        paired, pending = [], []
        for line in reversed(block_lines):
            if line.startswith(_RELAY_ATHLETE_SENTINEL):
                pending.append(line[len(_RELAY_ATHLETE_SENTINEL):])
            else:
                paired.append((line, list(reversed(pending))))
                pending = []
        paired.reverse()

        for result_line, athlete_names in paired:
            m = _PLACE_RE.match(result_line)
            if not m:
                continue
            place = m.group(1).rstrip("=")
            tokens = m.group(2).split()
            if not tokens or tokens[-1].upper() in NO_MARK:
                continue
            name, grade, team, mark = parse_row_fixed(result_line, current_cols, is_relay, is_field)
            if name is None or not mark:
                log_warning("Row parse failed", result_line.strip(), current_event)
                continue
            records.append({
                "event_name": current_event,
                "event_type": current_type,
                "place": place,
                "name": name,
                "grade": grade,
                "team": team,
                "mark": mark,
                "relay_athletes": athlete_names
            })

    for line in text.splitlines():
        stripped = line.strip()
        hdr = _EVENT_HEADER_RE.match(stripped)
        if hdr:
            flush()
            block_lines = []
            current_cols = None
            if _SKIP_SUBSECTION_RE.search(stripped):
                skip_block = True
                current_event = None
                continue

            raw_event = hdr.group(1).strip()
            round_label = (hdr.group(2) or "").strip().lower()
            mapped = normalize_event(raw_event)
            if mapped is None:
                log_warning("Unknown event skipped", raw_event, label)
                skip_block = True
                current_event = None
                continue

            skip_block = False
            current_event = mapped
            current_type = "Final" if (not round_label or "final" in round_label) else "Prelim"
            continue

        if not skip_block and current_event is not None and not _SEPARATOR_RE.match(stripped):
            cols = detect_columns(line)
            if cols is not None:
                current_cols = cols
                continue

        if current_event and not skip_block:
            if current_event in RELAY_EVENTS and _RELAY_ATHLETE_RE.search(line):
                for m in _RELAY_ATHLETE_RE.finditer(line):
                    name_str = m.group(1).strip()
                    if name_str:
                        block_lines.append(_RELAY_ATHLETE_SENTINEL + name_str)
                continue
            if not _PLACE_RE.match(line) and not _SEPARATOR_RE.match(stripped):
                continue
            block_lines.append(line)

    flush()
    return records

# -------------------------------------------------------------
# FORMAT B: TAB-DELIMITED (Above & Beyond Timing)
# -------------------------------------------------------------

_TAB_EVENT_HDR_RE = re.compile(
    r"^(?:Boys|Girls|Men|Women)\s+(?:Boys'?s?|Girls'?s?|Men'?s?|Women'?s?)\s+(.+?)$",
    re.IGNORECASE
)
_TAB_COL_HDR = "PLACE\tNAME\tGRADE"


def parse_format_b(text, label=""):
    records = []
    current_event = None
    current_type = None

    for line in text.splitlines():
        stripped = line.strip()
        if not stripped:
            continue

        hdr = _TAB_EVENT_HDR_RE.match(stripped)
        if hdr:
            raw_event = hdr.group(1).strip()
            mapped = normalize_event(raw_event)
            if mapped is None:
                log_warning("Unknown event skipped", raw_event, label)
                current_event = None
            else:
                current_event = mapped
                low = raw_event.lower()
                current_type = "Prelim" if "prelim" in low else "Final"
            continue

        if stripped.startswith(_TAB_COL_HDR):
            continue
        if current_event is None:
            continue

        parts = line.split("\t")
        if len(parts) < 6:
            continue

        place_raw = parts[0].strip()
        name = parts[1].strip()
        grade = parts[2].strip()
        team = parts[4].strip()
        mark = parts[5].strip()

        if not place_raw or not place_raw[0].isdigit():
            continue
        if not mark or mark.upper() in NO_MARK:
            continue

        is_relay = current_event in RELAY_EVENTS
        is_field = current_event in FIELD_EVENTS
        if is_field:
            mark = mark.rstrip("\".")

        if is_relay:
            records.append({
                "event_name": current_event, "event_type": current_type,
                "place": place_raw, "name": "", "grade": "",
                "team": team, "mark": mark, "relay_athletes": []
            })
        elif not is_relay and name:
            records.append({
                "event_name": current_event, "event_type": current_type,
                "place": place_raw, "name": name, "grade": grade,
                "team": team, "mark": mark, "relay_athletes": []
            })

    return records

# -------------------------------------------------------------
# FORMAT C: SPLIT EVENT/ROUND HEADERS
# -------------------------------------------------------------

_FMT_C_EVENT_RE = re.compile(r"^(?:Boys'?|Girls'?)\s+(.+?)$", re.IGNORECASE)
_FMT_C_ROUND_RE = re.compile(r"^(Finals?|Preliminaries?|Prelims?)$", re.IGNORECASE)
_FMT_C_COL_RE   = re.compile(r"\bNAME\b.*\bTEAM\b.*\bMARK\b", re.IGNORECASE)


def parse_format_c(text, label=""):
    records = []
    current_event = None
    current_type = None
    skip_block = False
    current_cols = None
    block_lines = []
    pending_event = None

    def flush():
        if not current_event or skip_block or current_cols is None:
            return
        is_relay = current_event in RELAY_EVENTS
        is_field = current_event in FIELD_EVENTS

        for result_line in block_lines:
            m = _PLACE_RE.match(result_line)
            if not m:
                continue
            place = m.group(1).rstrip("=")
            row = result_line.rstrip() + " " * 10
            all_tokens = [(mt.start(), mt.end(), mt.group()) for mt in re.finditer(r"\S+", result_line.rstrip())]
            tokens = [t for t in all_tokens if not _METRIC_RE.match(t[2])]
            tokens = _strip_trailing_heat_wind(tokens)
            mark_token = None
            for i in range(len(tokens) - 1, 0, -1):
                if tokens[i][0] - tokens[i - 1][1] >= _MARK_MIN_GAP:
                    mark_token = tokens[i]
                    break
            if mark_token is None:
                continue

            mark_val = mark_token[2]
            if mark_val.upper() in NO_MARK:
                continue
            if is_field:
                mark_val = mark_val.rstrip("\".")
            mark_start = mark_token[0]

            if is_relay:
                team = row[5:mark_start].strip()
                records.append({
                    "event_name": current_event, "event_type": current_type,
                    "place": place, "name": "", "grade": "",
                    "team": team, "mark": mark_val, "relay_athletes": []
                })
            else:
                c = current_cols
                name = row[c["name"]:c["yr"]].strip() if c["yr"] > 0 else row[5:mark_start].strip()
                grade = row[c["yr"]:c["team"]].strip() if c["yr"] > 0 else ""
                team = row[c["team"]:mark_start].strip()
                if grade.upper() not in {g.upper() for g in GRADES}:
                    grade = ""
                if not name or not mark_val:
                    continue
                records.append({
                    "event_name": current_event, "event_type": current_type,
                    "place": place, "name": name, "grade": grade,
                    "team": team, "mark": mark_val, "relay_athletes": []
                })

    for line in text.splitlines():
        stripped = line.strip()

        round_match = _FMT_C_ROUND_RE.match(stripped)
        if round_match:
            flush()
            block_lines = []
            current_cols = None
            if pending_event is not None:
                current_event = pending_event
                pending_event = None
            skip_block = False
            label_str = round_match.group(1).lower()
            current_type = "Prelim" if "prelim" in label_str else "Final"
            continue

        hdr = _FMT_C_EVENT_RE.match(stripped)
        if hdr and not _SEPARATOR_RE.match(stripped):
            if _SKIP_SUBSECTION_RE.search(stripped):
                flush()
                block_lines = []
                current_cols = None
                skip_block = True
                current_event = None
                pending_event = None
                continue

            raw_event = hdr.group(1).strip()
            raw_clean = re.sub(r"\s+(Finals?|Preliminaries?|Prelims?)$", "", raw_event, flags=re.IGNORECASE).strip()
            mapped = normalize_event(raw_clean)

            if mapped is None:
                log_warning("Unknown event skipped", raw_clean, label)
                flush()
                block_lines = []
                current_cols = None
                skip_block = True
                current_event = None
                pending_event = None
            else:
                flush()
                block_lines = []
                current_cols = None
                pending_event = mapped
                skip_block = True
            continue

        if _FMT_C_COL_RE.search(line):
            name_pos = line.upper().find("NAME")
            yr_pos = line.upper().find("YR")
            team_pos = line.upper().find("TEAM")
            current_cols = {
                "name": name_pos if name_pos >= 0 else 5,
                "yr":   yr_pos if yr_pos >= 0 else -1,
                "team": team_pos if team_pos >= 0 else 40,
            }
            continue

        if _SEPARATOR_RE.match(stripped):
            continue

        if current_event and not skip_block and _PLACE_RE.match(line):
            block_lines.append(line)

    flush()
    return records

# -------------------------------------------------------------
# FORMAT DETECTION + RAW PARSER
# -------------------------------------------------------------

_FORMAT_A_EVENT_WITH_ROUND_RE = re.compile(
    r"^(?:Boys'?|Girls'?)[-\s]+.+\s+(?:Finals?|Preliminaries?|Prelims?)\s*$",
    re.IGNORECASE
)


def detect_format(text):
    if ("PLACE\tNAME\tGRADE" in text
            or "Boys Boys " in text
            or "Girls Girls " in text
            or "Men Men's " in text
            or "Women Women's " in text):
        return "B"

    has_hdr_with_round = False
    has_standalone_round = False

    for line in text.splitlines():
        stripped = line.strip()
        if _FORMAT_A_EVENT_WITH_ROUND_RE.match(stripped):
            has_hdr_with_round = True
        if _FMT_C_ROUND_RE.match(stripped):
            has_standalone_round = True

    if has_standalone_round and not has_hdr_with_round:
        return "C"
    return "A"


def parse_raw_page(text, label=""):
    fmt = detect_format(text)
    if fmt == "B":
        return parse_format_b(text, label), fmt
    if fmt == "C":
        return parse_format_c(text, label), fmt
    return parse_format_a(text, label), fmt

# -------------------------------------------------------------
# FORMATTED API FALLBACK
# -------------------------------------------------------------

def _extract_meet_ids(meet_url, raw_url):
    """Pull numeric meetId and resultsId from the meet/raw URLs."""
    m1 = re.search(r"/meets/(\d+)", meet_url or "")
    m2 = re.search(r"/results/(\d+)", raw_url or "")
    return (m1.group(1) if m1 else None), (m2.group(1) if m2 else None)


def _extract_raw_result_candidates(soup):
    """Collect raw-result URL candidates from links on a page."""
    candidates = []
    seen = set()
    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href:
            continue

        match = re.search(r'(/meets/\d+[^\s"\']*/results/(\d+))(?:/(formatted|raw))?/?', href)
        if match:
            raw_url = urljoin(BASE, f"{match.group(1)}/raw")
            if raw_url not in seen:
                seen.add(raw_url)
                candidates.append(raw_url)
            continue

        if "/results" in href and "type=raw" in href:
            raw_url = href if href.startswith("http") else urljoin(BASE, href)
            if raw_url not in seen:
                seen.add(raw_url)
                candidates.append(raw_url)
    return candidates


def _expand_raw_candidates(seed_urls):
    """One-level expansion: open each seed URL and harvest nested result links."""
    expanded = []
    seen = set()

    def add(url):
        if url and url not in seen:
            seen.add(url)
            expanded.append(url)

    for u in seed_urls:
        add(u)

    for u in list(expanded):
        r = safe_get(u)
        if r is None:
            continue
        soup = BeautifulSoup(r.text, "html.parser")
        for discovered in _extract_raw_result_candidates(soup):
            add(discovered)

    return expanded


def _units_to_time(units):
    """Convert API millisecond units to a formatted time string."""
    try:
        seconds = int(units) / 1000.0
        mins = int(seconds // 60)
        secs = seconds - mins * 60
        return f"{mins}:{secs:05.2f}" if mins > 0 else f"{seconds:.2f}"
    except Exception:
        return None


def fetch_formatted_api_records(meet_id, results_id, label=""):
    """Call the MileSplit formatted API and return records in the same
    dict format used by the raw parsers. No relay athletes are available."""
    api_url = f"{BASE}/api/v1/meets/{meet_id}/performances"
    fields = (
        "id,meetId,eventName,roundName,place,mark,statusCode,"
        "firstName,lastName,teamName,gradYear,units,millimeters"
    )
    r = safe_get(api_url, params={"resultsId": str(results_id), "fields": fields})
    if r is None:
        log_warning("Formatted API fetch failed", label)
        return []

    try:
        payload = r.json()
    except Exception as e:
        log_warning("Formatted API JSON parse error", label, str(e))
        return []

    data = payload.get("data") if isinstance(payload, dict) else None
    if not isinstance(data, list):
        log_warning("Formatted API returned no data list", label)
        return []

    records = []
    for row in data:
        raw_event = (row.get("eventName") or "").strip()
        event_name = normalize_event(raw_event)
        if event_name is None:
            log_warning("Formatted API unknown event skipped", raw_event, label)
            continue

        round_str = (row.get("roundName") or "").strip().lower()
        event_type = "Prelim" if "prelim" in round_str else "Final"
        place = str(row.get("place") or "").strip()
        if not place:
            continue

        status = (row.get("statusCode") or "").strip()
        mark = (row.get("mark") or "").strip()
        if not mark and status:
            mark = status
        if not mark and event_name not in FIELD_EVENTS:
            mark = _units_to_time(row.get("units")) or ""
        if not mark or mark.upper() in NO_MARK:
            continue
        if event_name in FIELD_EVENTS:
            mark = mark.rstrip("\".")

        first = (row.get("firstName") or "").strip()
        last = (row.get("lastName") or "").strip()
        name = f"{first} {last}".strip()
        team = (row.get("teamName") or "").strip()

        grade = ""
        try:
            offset = int(row.get("gradYear") or 9999) - YEAR
            grade = {0: "SR", 1: "JR", 2: "SO", 3: "FR"}.get(offset, "")
        except Exception:
            pass

        is_relay = event_name in RELAY_EVENTS
        if is_relay:
            if not team:
                continue
            records.append({
                "event_name": event_name, "event_type": event_type,
                "place": place, "name": "", "grade": "",
                "team": team, "mark": mark, "relay_athletes": []
            })
        else:
            if not name or not team:
                continue
            records.append({
                "event_name": event_name, "event_type": event_type,
                "place": place, "name": name, "grade": grade,
                "team": team, "mark": mark, "relay_athletes": []
            })

    return records

# -------------------------------------------------------------
# ATHLETE LOOKUP / INSERT
# -------------------------------------------------------------

_athlete_cache = {}


def process_athlete(full_name, school_id, grad_year):
    parts = full_name.strip().split()
    if not parts:
        return None

    first = parts[0].upper()
    last = " ".join(parts[1:]).upper() if len(parts) > 1 else ""
    key = (first, last, school_id, grad_year)

    if key in _athlete_cache:
        return _athlete_cache[key]

    athlete_id = db.get_athlete_id(first, last, school_id, grad_year)
    if athlete_id is None:
        athlete_id = db.get_athlete_id_by_name_school(first, last, school_id)
        if athlete_id is not None:
            existing_grad = db.get_athlete_grad_year(athlete_id)
            if grad_year != 9999 and existing_grad != grad_year:
                db.update_athlete_grad_year(athlete_id, grad_year, commit=False)
                log_warning("Grad year updated", f"{first} {last}", f"school_id={school_id} old={existing_grad} new={grad_year}")
        else:
            athlete_id = db.insert_athlete(school_id, first, last, GENDER, grad_year, commit=False)
            log_warning("Athlete created", f"{first} {last}", f"school_id={school_id} grad={grad_year}")

    _athlete_cache[key] = athlete_id
    return athlete_id

# -------------------------------------------------------------
# GET RAW URL + HOST + MATCH DEBUG
# -------------------------------------------------------------

def get_meet_raw_url(meet_type, meet_num):
    mt = (meet_type or "").strip().lower()

    # 2026+ uses IHSAA round pages as the primary meet-link source.
    if YEAR >= IHSAA_FROM_YEAR:
        cache_key = (YEAR, GENDER, mt)
        if cache_key not in _ihsaa_meet_cache:
            _ihsaa_meet_cache[cache_key] = get_ihsaa_meet_links(meet_type, GENDER, YEAR)

        ihsaa_map = _ihsaa_meet_cache.get(cache_key, {})
        from_ihsaa = ihsaa_map.get(int(meet_num))
        if from_ihsaa:
            return _build_meet_info_from_meet_url(
                from_ihsaa["meet_url"],
                host=from_ihsaa.get("host"),
                anchor_text=from_ihsaa.get("anchor_text", ""),
                source="IHSAA"
            )

        log_warning("IHSAA meet link missing; fallback to MileSplit search", f"{GENDER} {meet_type} {meet_num}", _ihsaa_tournament_url(YEAR, GENDER, meet_type))

    selected = None

    for month in INDEX_MONTHS:
        # Start with page 1
        page = 1
        max_pages = 5  # Reasonable limit to avoid infinite loops

        while page <= max_pages:
            index_url = f"{BASE}/results?month={month}&year={YEAR}&level=hs&page={page}"
            r = safe_get(index_url)
            if r is None:
                break  # Can't reach this page, stop pagination

            soup = BeautifulSoup(r.text, "html.parser")

            for a in soup.find_all("a", href=True):
                text = a.get_text(" ", strip=True)
                href = a.get("href", "").strip()
                if not text or "/meets/" not in href:
                    continue

                if not _is_target_meet(text, meet_type, meet_num, GENDER):
                    continue

                # MATCH FOUND!
                meet_url = href if href.startswith("http") else urljoin(BASE, href)
                selected = {
                    "anchor_text": text,
                    "meet_url": meet_url,
                    "source": "MileSplit"
                }
                break

            if selected is not None:
                break

            # Check if there's a next page button
            next_page_link = soup.find("a", string=re.compile(r"next", re.IGNORECASE))
            if not next_page_link:
                # No next button found, stop pagination
                break

            page += 1

        if selected is not None:
            break

    if selected is None:
        return {
            "raw_url": None,
            "raw_candidates": [],
            "host": None,
            "meet_url": None,
            "anchor_text": "",
            "raw_option_found": False,
            "source": "MileSplit"
        }

    meet_url = selected["meet_url"]
    anchor_text = selected["anchor_text"]
    host = anchor_text.split("-")[-1].strip() if "-" in anchor_text else anchor_text.strip()

    return _build_meet_info_from_meet_url(meet_url, host=host, anchor_text=anchor_text, source="MileSplit")

# -------------------------------------------------------------
# DETERMINE MEETS TO PROCESS
# -------------------------------------------------------------

if MEET_TYPE == "Sectional":
    meets_to_process = SEC_TO_PROCESS
elif MEET_TYPE == "Regional":
    meets_to_process = REG_TO_PROCESS
else:
    meets_to_process = [1]  # State has one meet per gender

# -------------------------------------------------------------
# MAIN
# -------------------------------------------------------------

print("MileSplit raw scraper started.")
start_time = datetime.now()
print(f"Start time: {start_time.strftime('%H:%M:%S')}")

for meet_num in meets_to_process:
    label = f"{YEAR} {GENDER} {MEET_TYPE}" + (f" {meet_num}" if MEET_TYPE != "State" else "")
    print(f"Processing {label}...")

    meet_id = db.get_meet_id(MEET_TYPE, meet_num, YEAR, GENDER)
    if meet_id is not None:
        print(f"  Meet already exists in meet table (meet_id={meet_id}); skipping scrape.")
        print("  Rows read in: 0")
        runStats.append({
            "label": label,
            "source": "",
            "anchor_text": "",
            "meet_url": "",
            "raw_url": "",
            "raw_option_found": 0,
            "parser_format": "",
            "parsed_rows": 0,
            "athlete_rows_inserted": 0,
            "relay_rows_inserted": 0,
            "status": "meet_exists_skipped",
            "note": f"Meet already exists in meet table (meet_id={meet_id})"
        })
        continue

    meet_info = get_meet_raw_url(MEET_TYPE, meet_num)
    raw_url = meet_info["raw_url"]
    raw_candidates = meet_info.get("raw_candidates") or ([raw_url] if raw_url else [])
    meet_host = meet_info["host"]

    meet_stat = {
        "label": label,
        "source": meet_info.get("source", ""),
        "anchor_text": meet_info.get("anchor_text", ""),
        "meet_url": meet_info.get("meet_url", ""),
        "raw_url": raw_url or "",
        "raw_option_found": int(bool(meet_info.get("raw_option_found"))),
        "parser_format": "",
        "parsed_rows": 0,
        "athlete_rows_inserted": 0,
        "relay_rows_inserted": 0,
        "status": "ok",
        "note": ""
    }

    if raw_url is None:
        log_warning("Meet URL not found", label)
        print(raw_url)
        print("  Rows read in: 0")
        meet_stat["status"] = "meet_not_found"
        meet_stat["note"] = "No matching meet link in monthly results index"
        runStats.append(meet_stat)
        continue

    print(f"  RAW URL: {raw_url}")

    records = []
    fmt = ""
    successful_raw_url = None
    last_parse_error = None
    any_fetch_succeeded = False

    for candidate_url in raw_candidates:
        r = safe_get(candidate_url)
        if r is None:
            continue

        any_fetch_succeeded = True
        if successful_raw_url is None:
            successful_raw_url = candidate_url  # mark first reachable URL

        soup = BeautifulSoup(r.text, "html.parser")
        pre = soup.find("pre") or soup.find("code")
        raw_text = pre.get_text() if pre else r.text

        try:
            candidate_records, candidate_fmt = parse_raw_page(raw_text, label=label)
        except Exception as e:
            last_parse_error = str(e)
            log_warning("Parse error", candidate_url, str(e))
            continue

        if len(candidate_records) > len(records):
            records = candidate_records
            fmt = candidate_fmt
            successful_raw_url = candidate_url

    if not any_fetch_succeeded:
        # No candidate URL could be reached at all
        meet_stat["status"] = "raw_fetch_failed"
        meet_stat["note"] = "Could not fetch any raw result URL"
        print("  Rows read in: 0")
        runStats.append(meet_stat)
        continue
    elif last_parse_error and not records:
        # Every reachable candidate threw a parse error
        meet_stat["status"] = "parse_error"
        meet_stat["note"] = last_parse_error
        print("  Rows read in: 0")
        runStats.append(meet_stat)
        continue

    # URL was reachable; may have 0 rows (formatted API fallback will handle that)
    raw_url = successful_raw_url
    meet_stat["raw_url"] = raw_url
    meet_stat["parser_format"] = fmt

    parsed_count = len(records)
    meet_stat["parsed_rows"] = parsed_count
    if parsed_count > 0:
        print(f"  Parsed {parsed_count} result rows. Format {meet_stat['parser_format']}")

    if parsed_count < MIN_ROWS_EXPECTED:
        meet_stat["note"] = f"Parsed rows below threshold ({parsed_count} < {MIN_ROWS_EXPECTED})"

    # ------------------------------------------------------------------
    # FORMATTED API FALLBACK: fires only when raw yields 0 rows
    # Note: relay athlete names will be empty for API-fallback meets
    # ------------------------------------------------------------------
    if parsed_count == 0:
        api_meet_id, _ = _extract_meet_ids(meet_info.get("meet_url"), raw_url)
        best_records = []
        best_candidate_url = raw_url

        if api_meet_id:
            for candidate_url in raw_candidates:
                _, api_results_id = _extract_meet_ids(meet_info.get("meet_url"), candidate_url)
                if not api_results_id:
                    continue
                candidate_api_records = fetch_formatted_api_records(api_meet_id, api_results_id, label=label)
                if len(candidate_api_records) > len(best_records):
                    best_records = candidate_api_records
                    best_candidate_url = candidate_url

        if best_records:
            records = best_records
            parsed_count = len(records)
            raw_url = best_candidate_url
            meet_stat["raw_url"] = raw_url
            meet_stat["parser_format"] = "formatted_api"
            meet_stat["parsed_rows"] = parsed_count
            meet_stat["note"] = f"formatted_api fallback: {parsed_count} rows"
        else:
            log_warning("Formatted API fallback found no rows", label)

    print(f"  Final parsed rows: {parsed_count}. Source: {meet_stat['parser_format'] or 'raw'}")
    print(f"  Rows read in: {parsed_count}")

    if parsed_count > 0 and meet_id is None:
        db.insert_meet(meet_host or label, MEET_TYPE, meet_num, YEAR, GENDER, commit=False)
        meet_id = db.get_meet_id(MEET_TYPE, meet_num, YEAR, GENDER)

    for rec in records:
        event_name = rec["event_name"]
        event_type = rec["event_type"]
        place = rec["place"]
        mark = rec["mark"]
        team = team_mapping(rec["team"])
        grade = rec["grade"]

        if not team:
            log_warning("Empty team name - skipping", rec.get("name", ""), label)
            continue

        is_relay = event_name in RELAY_EVENTS

        try:
            mark2 = (convert.distance_to_inches(mark)
                     if event_name in FIELD_EVENTS
                     else convert.time_to_seconds(mark))
        except Exception:
            mark2 = None

        school_id = db.get_school_id(team)
        if school_id is None:
            # API returns full names like "Ben Davis High School"; try stripping suffix
            stripped_team = re.sub(r"\s+High School$", "", team, flags=re.IGNORECASE).strip()
            if stripped_team != team:
                school_id = db.get_school_id(stripped_team)
        if school_id is None:
            log_warning("School not found", team, label)
            continue

        if is_relay:
            if db.get_relay_result(school_id, meet_id, event_name) is None:
                db.insert_relay_result(school_id, meet_id, event_name, mark, mark2, place, "", commit=False)
                meet_stat["relay_rows_inserted"] += 1

            relay_athletes = rec.get("relay_athletes", [])
            if relay_athletes:
                relay_id = db.get_relay_result(school_id, meet_id, event_name)
                if relay_id is not None:
                    for full_name in relay_athletes:
                        parts = full_name.strip().split()
                        if len(parts) < 2:
                            continue
                        r_first = parts[0].upper()
                        r_last = " ".join(parts[1:]).upper()
                        athlete_id = db.get_athlete_id_by_name_school(r_first, r_last, school_id)
                        if athlete_id is not None:
                            try:
                                db.insert_relay_athlete(relay_id, athlete_id, commit=False)
                            except Exception:
                                pass
        else:
            if not grade:
                parts = rec["name"].strip().split()
                first = parts[0].upper() if parts else ""
                last = " ".join(parts[1:]).upper() if len(parts) > 1 else ""
                existing = db.get_athlete_id_by_name_school(first, last, school_id) if first else None
                if existing is not None:
                    existing_grad = db.get_athlete_grad_year(existing)
                    offset = existing_grad - YEAR
                    grade = {0: "SR", 1: "JR", 2: "SO", 3: "FR"}.get(offset, "")
                    if not grade:
                        log_warning("Grade missing, not recoverable", rec["name"], f"school_id={school_id}")
                else:
                    log_warning("Grade missing, athlete not in DB", rec["name"], f"school_id={school_id}")

            grad_year = calculate_grad_year(grade) if grade else 9999
            athlete_id = process_athlete(rec["name"], school_id, grad_year)
            if athlete_id is None:
                continue

            norm_grade = grade_normalization(grade) if grade else "Unknown"
            if db.get_athlete_result(athlete_id, meet_id, event_name, event_type) is None:
                db.insert_athlete_result(athlete_id, meet_id, event_name, event_type, norm_grade, mark, mark2, place, commit=False)
                meet_stat["athlete_rows_inserted"] += 1

    runStats.append(meet_stat)
    gc.collect()

db.do_commit()

end_time = datetime.now()
print(f"End time:     {end_time.strftime('%H:%M:%S')}")
print(f"Elapsed time: {end_time - start_time}")
total_rows_read = sum(int(s.get("parsed_rows", 0) or 0) for s in runStats)
print(f"Total rows read in: {total_rows_read}")

warnings_file = f"{YEAR} {GENDER} {MEET_TYPE} MileSplit Warnings.csv"
warningDF.to_csv(warnings_file, index=False)
print(f"Warnings written to: {warnings_file}")

stats_file = f"{YEAR} {GENDER} {MEET_TYPE} MileSplit Run Summary.csv"
pd.DataFrame(runStats).to_csv(stats_file, index=False)
print(f"Run summary written to: {stats_file}")
print("Done!")


Keeping current Track.db (no restore).
Active DB: ../web/data/Track.db
MileSplit raw scraper started.
Start time: 00:07:31
Processing 2026 Girls Sectional 1...
  Meet already exists in meet table (meet_id=3225); skipping scrape.
  Rows read in: 0
Processing 2026 Girls Sectional 2...
  Meet already exists in meet table (meet_id=3226); skipping scrape.
  Rows read in: 0
Processing 2026 Girls Sectional 3...
  Meet already exists in meet table (meet_id=3227); skipping scrape.
  Rows read in: 0
Processing 2026 Girls Sectional 4...
  Meet already exists in meet table (meet_id=3240); skipping scrape.
  Rows read in: 0
Processing 2026 Girls Sectional 5...
None
  Rows read in: 0
Processing 2026 Girls Sectional 6...
None
  Rows read in: 0
Processing 2026 Girls Sectional 7...
  Meet already exists in meet table (meet_id=3228); skipping scrape.
  Rows read in: 0
Processing 2026 Girls Sectional 8...
  Meet already exists in meet table (meet_id=3229); skipping scrape.
  Rows read in: 0
Processing 20